# FSDP Distributed Training (Single-GPU Demo)

Companion notebook for the [Distributed Training](https://codemia.io/courses/applied_ai_systems/distributed_training) lesson on Codemia.

**What you will build**: a walkthrough of Fully Sharded Data Parallel (FSDP) in PyTorch — the configuration, the memory accounting, the interaction with gradient checkpointing — running in **single-GPU demonstration mode** so it works on free Colab. The notebook also explains exactly what changes when you scale to multiple GPUs.

**Why FSDP matters**: vanilla DDP replicates the full model and optimizer state on every GPU, so the biggest model you can train is bounded by a single GPU's memory. FSDP shards the parameters, gradients, and optimizer state across GPUs, gathering only the shard needed for the current forward/backward step. For a 7B model, the optimizer state alone (Adam keeps 2 moments of fp32 = 56 GB) is too big for a single A100. With FSDP across 8 GPUs, that drops to 7 GB per GPU.

**References**:
- [PyTorch FSDP tutorial](https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html)
- [ZeRO paper](https://arxiv.org/abs/1910.02054) — DeepSpeed's original version of the same idea
- [PyTorch FSDP blog](https://pytorch.org/blog/introducing-pytorch-fully-sharded-data-parallel-api/)

**Time**: ~40 minutes  **GPU required**: yes (T4 works for the demo)

## Important: single-GPU mode

FSDP is designed for **multi-GPU** training. Free Colab only has one GPU, so this notebook runs FSDP with `world_size=1` — which still exercises the real FSDP API and the actual sharding logic, but produces no speedup and reports sharding splits that would span multiple ranks in a real deployment.

At every step where multi-GPU matters, we print a note about what the numbers would look like with, say, 4 GPUs.

## Setup

In [ ]:
%pip install -q torch

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp import ShardingStrategy, MixedPrecision
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
from torch.utils.checkpoint import checkpoint_sequential
import functools

print('torch:', torch.__version__)
assert torch.cuda.is_available(), 'FSDP requires a GPU.'

In [ ]:
# FSDP uses torch.distributed collectives (all_gather, reduce_scatter, etc.) even
# when there is only one rank. We have to initialize a process group — any backend
# and any rendezvous method will do as long as world_size matches.
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12355'
os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '1'

if not dist.is_initialized():
    dist.init_process_group(backend='nccl', rank=0, world_size=1)

rank = dist.get_rank()
world_size = dist.get_world_size()
print(f'rank={rank} world_size={world_size}')

## Concept recap

Three things get sharded under FSDP's `FULL_SHARD` strategy:

1. **Parameters** — each rank holds `1/N` of each layer's weights at rest.
2. **Gradients** — the backward pass produces sharded gradients (via reduce-scatter instead of all-reduce).
3. **Optimizer states** — Adam stores m and v; they live on the rank that owns the parameter shard.

At forward time, FSDP does an `all_gather` to reconstruct the full weight for the current FSDP unit, runs the computation, then discards the gathered weight. The memory peak is therefore `max(one_unit_size, activation_size)` rather than `total_model_size`.

**Auto-wrap policy** decides how to group layers into FSDP units. A size-based policy says "group layers until accumulated params > threshold, then make that a unit". For transformers you usually wrap by block (one unit per transformer layer).

## Implementation

We build a toy transformer-like model to stand in for the real thing. It has repeated linear blocks so sharding actually has something to group.

In [ ]:
class TinyBlock(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden)
        self.fc2 = nn.Linear(hidden, dim)
        self.ln = nn.LayerNorm(dim)

    def forward(self, x):
        # Pre-norm residual, like GPT-2/Llama style blocks.
        h = self.ln(x)
        h = torch.relu(self.fc1(h))
        h = self.fc2(h)
        return x + h

class TinyModel(nn.Module):
    def __init__(self, dim=512, hidden=2048, n_blocks=8, vocab=1000):
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.blocks = nn.ModuleList([TinyBlock(dim, hidden) for _ in range(n_blocks)])
        self.head = nn.Linear(dim, vocab)

    def forward(self, ids):
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x)
        return self.head(x)

In [ ]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

# Build once on CPU so we can count parameters and compare memory.
base = TinyModel(dim=512, hidden=2048, n_blocks=8)
total_params = count_params(base)
param_bytes = total_params * 4  # fp32
print(f'model: {total_params/1e6:.2f}M params')
print(f'fp32 weight memory (baseline): {param_bytes/1e6:.1f} MB')
# Adam optimizer state: 8 bytes/param (two fp32 moments).
print(f'adam optimizer state: {total_params * 8 / 1e6:.1f} MB')
del base

## Experiment 1: baseline DDP-equivalent (full replication)

We train once with no FSDP — just the model moved to the GPU. Measure peak memory and per-step time.

In [ ]:
def make_batch(batch_size=16, seq_len=64, vocab=1000, device='cuda'):
    return torch.randint(0, vocab, (batch_size, seq_len), device=device)

def train_one_step(model, optimizer, batch):
    optimizer.zero_grad()
    logits = model(batch)
    # Language modeling style loss against shifted inputs.
    loss = nn.functional.cross_entropy(
        logits[:, :-1].reshape(-1, logits.size(-1)),
        batch[:, 1:].reshape(-1),
    )
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
def measure(model_factory, n_steps=10, label=''):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    model = model_factory().cuda()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    batch = make_batch()

    # Warm up.
    for _ in range(2):
        train_one_step(model, optimizer, batch)
    torch.cuda.synchronize()

    # Time n steps.
    t0 = time.time()
    for _ in range(n_steps):
        train_one_step(model, optimizer, batch)
    torch.cuda.synchronize()
    step_time = (time.time() - t0) / n_steps

    peak_mb = torch.cuda.max_memory_allocated() / 1e6
    print(f'{label:<24} peak_mem={peak_mb:7.1f} MB  step_time={step_time*1000:6.1f} ms')

    del model, optimizer
    torch.cuda.empty_cache()
    return peak_mb, step_time

baseline_mem, baseline_time = measure(
    lambda: TinyModel(dim=512, hidden=2048, n_blocks=8),
    label='baseline (no FSDP)',
)

## Experiment 2: FSDP full-shard

We wrap the model with FSDP and `FULL_SHARD` strategy. On single-GPU the memory savings should be near zero (there's nothing to split across ranks), but the wrapping and the shard-gather-run-reshard flow still happens, and the code is identical to what you'd use on 8 GPUs.

In [ ]:
def make_fsdp_model():
    model = TinyModel(dim=512, hidden=2048, n_blocks=8)
    # Auto-wrap: group layers into FSDP units until each unit exceeds 1e5 params.
    # In a real transformer you'd use `transformer_auto_wrap_policy` with the
    # block class as the wrap target, which makes each transformer layer one unit.
    wrap_policy = functools.partial(
        size_based_auto_wrap_policy,
        min_num_params=100_000,
    )
    model = model.cuda()
    return FSDP(
        model,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        auto_wrap_policy=wrap_policy,
        # mixed_precision=MixedPrecision(param_dtype=torch.float16, ...) would cut
        # memory and speed up matmul on A100+. We leave it off for simplicity.
        device_id=torch.cuda.current_device(),
    )

fsdp_mem, fsdp_time = measure(make_fsdp_model, label='FSDP FULL_SHARD')
print()
print(f'single-GPU memory delta: {fsdp_mem/baseline_mem:.2%}')
print()
print('NOTE: with world_size=N>1, FSDP would divide param+grad+opt state memory')
print('by N. With 4 ranks on the same model, you would see roughly 1/4 the')
print('parameter and optimizer memory per rank.')

### What FSDP actually bought you

Let me break down the memory accounting to show where the savings come from. For an `N`-rank full-shard setup, the per-rank memory for a model with `P` parameters looks like:

| component      | baseline (DDP)  | FSDP FULL_SHARD |
|----------------|-----------------|-----------------|
| parameters     | `4P` bytes      | `4P/N` bytes    |
| gradients      | `4P` bytes      | `4P/N` bytes    |
| adam state     | `8P` bytes      | `8P/N` bytes    |
| activations    | depends         | same            |
| temp gather    | 0               | `4*max_unit`    |

The first three rows all divide by N. The `temp gather` overhead is the biggest single FSDP unit, because that's what the all_gather brings into memory transiently. Picking a good wrap policy keeps `max_unit` small.

## Experiment 3: FSDP + gradient checkpointing

Activation memory is the *other* big thing you can shrink. Gradient checkpointing (a.k.a. activation checkpointing) drops stored activations during the forward pass and recomputes them during the backward pass. Tradeoff: ~30% slower, ~1.5-3x activation memory savings.

In [ ]:
from torch.distributed.algorithms._checkpoint.checkpoint_wrapper import (
    checkpoint_wrapper,
    CheckpointImpl,
    apply_activation_checkpointing,
)

def make_fsdp_ckpt_model():
    model = TinyModel(dim=512, hidden=2048, n_blocks=8)
    model = model.cuda()
    model = FSDP(
        model,
        sharding_strategy=ShardingStrategy.FULL_SHARD,
        auto_wrap_policy=functools.partial(
            size_based_auto_wrap_policy, min_num_params=100_000
        ),
        device_id=torch.cuda.current_device(),
    )
    # Wrap every TinyBlock with the checkpoint wrapper. During forward, the block
    # runs in torch.no_grad mode and stashes only the input. During backward,
    # PyTorch reruns the forward pass to get the intermediate activations back.
    apply_activation_checkpointing(
        model,
        checkpoint_wrapper_fn=functools.partial(
            checkpoint_wrapper, checkpoint_impl=CheckpointImpl.NO_REENTRANT
        ),
        check_fn=lambda submodule: isinstance(submodule, TinyBlock),
    )
    return model

ckpt_mem, ckpt_time = measure(make_fsdp_ckpt_model, label='FSDP + checkpoint')
print()
print(f'activation memory saved vs plain FSDP: ~{(fsdp_mem-ckpt_mem)/fsdp_mem:.1%}')
print(f'step time penalty:                     ~{(ckpt_time-fsdp_time)/fsdp_time:.1%}')

## Multi-GPU: what actually changes

Moving from this single-GPU demo to an 8-GPU training run requires the following changes:

```python
# 1. Launch with torchrun instead of running the notebook directly.
# torchrun --nproc_per_node=8 train.py
# 2. Read rank + world_size from env, not hard-coded to 0/1:
rank = int(os.environ['RANK'])
world_size = int(os.environ['WORLD_SIZE'])
dist.init_process_group('nccl', rank=rank, world_size=world_size)
torch.cuda.set_device(rank)
# 3. Use a DistributedSampler so each rank sees a different slice of data:
sampler = torch.utils.data.distributed.DistributedSampler(dataset, num_replicas=world_size, rank=rank)
# 4. The FSDP() wrapper call is literally unchanged.
# 5. Save checkpoints using FSDP state_dict type config:
from torch.distributed.fsdp import StateDictType, FullStateDictConfig
with FSDP.state_dict_type(model, StateDictType.FULL_STATE_DICT,
                          FullStateDictConfig(offload_to_cpu=True, rank0_only=True)):
    state = model.state_dict()
    if rank == 0:
        torch.save(state, 'ckpt.pt')
```

Everything else — the model code, the training loop, the loss computation — stays identical. That is FSDP's selling point: sharding is transparent.

## Variations

Try `SHARDING_STRATEGY.SHARD_GRAD_OP` (sometimes called ZeRO-2): shards gradients and optimizer state but keeps params replicated. Good when params are small relative to optimizer state (which they are with Adam at fp32).

In [ ]:
def make_fsdp_zero2():
    model = TinyModel(dim=512, hidden=2048, n_blocks=8).cuda()
    return FSDP(
        model,
        sharding_strategy=ShardingStrategy.SHARD_GRAD_OP,  # ZeRO-2 style
        auto_wrap_policy=functools.partial(
            size_based_auto_wrap_policy, min_num_params=100_000
        ),
        device_id=torch.cuda.current_device(),
    )

zero2_mem, zero2_time = measure(make_fsdp_zero2, label='FSDP SHARD_GRAD_OP')

print()
print('summary (single GPU; multi-GPU would divide sharded pieces by N):')
print(f'  baseline             {baseline_mem:7.1f} MB  {baseline_time*1000:6.1f} ms/step')
print(f'  FSDP FULL_SHARD      {fsdp_mem:7.1f} MB  {fsdp_time*1000:6.1f} ms/step')
print(f'  FSDP SHARD_GRAD_OP   {zero2_mem:7.1f} MB  {zero2_time*1000:6.1f} ms/step')
print(f'  FSDP + checkpoint    {ckpt_mem:7.1f} MB  {ckpt_time*1000:6.1f} ms/step')

In [ ]:
# Clean up the process group so subsequent notebook runs don't collide.
if dist.is_initialized():
    dist.destroy_process_group()
print('process group destroyed')

## What to try next

- Run this notebook under `torchrun --nproc_per_node=2` on a machine with 2 GPUs (Colab Pro, Lambda, etc.) and watch the memory per rank drop roughly in half.
- Enable mixed precision: `MixedPrecision(param_dtype=torch.float16, reduce_dtype=torch.float16, buffer_dtype=torch.float16)` cuts parameter memory by 2x and speeds up matmul on A100+. Note: gradients need to stay in fp32 for numerical stability — use `reduce_dtype=fp32` if you see NaNs.
- Swap the toy model for a real one: a small Llama-style transformer (GPT-2 is a good starting point). Use `transformer_auto_wrap_policy` with the transformer block class to wrap each block separately.
- Profile the collective calls with `torch.profiler`. You will see `ncclAllGather` and `ncclReduceScatter` in the trace — that's the actual data movement FSDP schedules.
- Try `HYBRID_SHARD`: shards within a node, replicates across nodes. Useful for multi-node training where intra-node NVLink is much faster than inter-node Infiniband.